# **Teen Mental Health Risk Analyzer — Social Media Impact Study**

This notebook analyzes the impact of social media on teenager mental health using machine learning models. Two predictive models are implemented:

   1. Mental Health Risk Prediction Model: predicts individual teenager risk levels (Low, Medium, or High) based on social media usage patterns, sleep habits, platform behavior, and lifestyle metrics.
   2. Mental Health Index Prediction Model: estimates a continuous mental health score for each teenager based on anxiety, depression, and stress indicators combined with daily screen time and demographic features.

# **Stage 1: Retrieving and loading the dataset**

As the first step, we load the dataset and provide some initial overview on the data through shape, columns and info methods

In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [43]:
# Load the dataset
file_path = '../dataset/Teen_Mental_Health_Dataset.csv'
data = pd.read_csv(file_path);

In [44]:
print(data.head())

   age  gender  daily_social_media_hours platform_usage  sleep_hours  \
0   14    male                       7.9      Instagram          7.4   
1   19  female                       1.9         TikTok          8.0   
2   17  female                       1.3      Instagram          7.6   
3   15    male                       7.4         TikTok          6.9   
4   15  female                       4.7           Both          4.9   

   screen_time_before_sleep  academic_performance  physical_activity  \
0                       2.9                  3.01                1.5   
1                       2.9                  3.22                0.8   
2                       0.5                  3.92                0.0   
3                       1.6                  3.48                0.8   
4                       3.0                  2.37                1.4   

  social_interaction_level  stress_level  anxiety_level  addiction_level  \
0                      low             2              2   

In [45]:
print(f"Shape of the data (rows, columns): {data.shape}")
print(f"Columns in data: {list(data.columns)}\n")

data.info()

Shape of the data (rows, columns): (1200, 13)
Columns in data: ['age', 'gender', 'daily_social_media_hours', 'platform_usage', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level', 'depression_label']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       1200 non-null   int64  
 1   gender                    1200 non-null   object 
 2   daily_social_media_hours  1200 non-null   float64
 3   platform_usage            1200 non-null   object 
 4   sleep_hours               1200 non-null   float64
 5   screen_time_before_sleep  1200 non-null   float64
 6   academic_performance      1200 non-null   float64
 7   physical_activity         1200 non-null   float64
 8   social_interaction_lev

#### Initial Data Quality Assessment

- The dataset contains a mix of numerical and categorical features, so encoding will be required for columns such as `gender`, `platform`, and `usage_frequency`.
- No missing values were found across any column.
- Minor inconsistencies in categorical labels (e.g. casing differences) were identified and will be addressed during cleaning.
- The dataset is structured and ready for preprocessing.

This confirms the data is complete and suitable for analysis after encoding and label standardization.

## **Stage 2: Data Preparation**
In this section, we prepare the dataset for teenager mental health risk prediction.
This includes:
1. Data Cleansing : removing duplicates, inconsistent categorical labels, and invalid or out-of-range values.
2. Data Transformation : feature engineering (mental health index, risk labels, sleep deficit), categorical encoding, and feature scaling.

### 2.1 Data Cleansing

In [46]:
duplicates = data.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    df = data.drop_duplicates()
    print("Duplicates removed.")
else:
    print("No duplicates found.")

Number of duplicate rows: 0
No duplicates found.


No duplicates were found in the dataset.

In [47]:
missing_values = data.isnull().sum();
print(f'Number of missing values in each column:\n{missing_values}')

Number of missing values in each column:
age                         0
gender                      0
daily_social_media_hours    0
platform_usage              0
sleep_hours                 0
screen_time_before_sleep    0
academic_performance        0
physical_activity           0
social_interaction_level    0
stress_level                0
anxiety_level               0
addiction_level             0
depression_label            0
dtype: int64


No missing values were found in dataset, indicating completeness and suitability for modeling

In [48]:
invalid_age = data[(data['age'] < 10) | (data['age'] > 20)].shape[0]
invalid_daily_hours = data[(data['daily_social_media_hours'] < 0) | (data['daily_social_media_hours'] > 16)].shape[0]
invalid_sleep_hours = data[(data['sleep_hours'] <= 0) | (data['sleep_hours'] > 14)].shape[0]
invalid_anxiety = data[(data['anxiety_level'] < 0) | (data['anxiety_level'] > 10)].shape[0]
invalid_depression = data[(data['depression_label'] < 0) | (data['depression_label'] > 1)].shape[0]
invalid_stress = data[(data['stress_level'] < 0) | (data['stress_level'] > 10)].shape[0]
invalid_academic = data[(data['academic_performance'] < 0) | (data['academic_performance'] > 100)].shape[0]
invalid_screen_before_sleep = data[(data['screen_time_before_sleep'] < 0) | (data['screen_time_before_sleep'] > 12)].shape[0]
invalid_addiction_level = data[(data['addiction_level'] < 0) | (data['addiction_level'] > 10)].shape[0]

print(f"Invalid Age values: {invalid_age}")
print(f"Invalid Daily Usage Hours: {invalid_daily_hours}")
print(f"Invalid Sleep Hours: {invalid_sleep_hours}")
print(f"Invalid Anxiety Score: {invalid_anxiety}")
print(f"Invalid Depression Label: {invalid_depression}")
print(f"Invalid Stress Score: {invalid_stress}")
print(f"Invalid Academic Performance: {invalid_academic}")
print(f"Invalid Screen Time Before Sleep: {invalid_screen_before_sleep}")
print(f"Invalid Addiction Level: {invalid_addiction_level}")

Invalid Age values: 0
Invalid Daily Usage Hours: 0
Invalid Sleep Hours: 0
Invalid Anxiety Score: 0
Invalid Depression Label: 0
Invalid Stress Score: 0
Invalid Academic Performance: 0
Invalid Screen Time Before Sleep: 0
Invalid Addiction Level: 0


All the values lies within the resonable ranges. This confirms the dataset is clean and ready for features extraction and engineering.

### 2.2 Features Engineering

We create new features that may improve model performance.

In [49]:
# Mental health index (regression target)
score_cols = ['anxiety_level', 'depression_label', 'stress_level']
data['MH_Index'] = data[score_cols].mean(axis=1).round(2)

# Risk label (classification target)
def risk_label(score):
    if score >= 7:   return 'High'
    elif score >= 4: return 'Medium'
    else:            return 'Low'

data['Risk_Level'] = data['MH_Index'].apply(risk_label)

# Sleep deficit feature
data['Sleep_Deficit'] = (8.5 - data['sleep_hours']).clip(lower=0).round(2)

# Sleep deprived flag
data['Sleep_Deprived'] = (data['sleep_hours'] < 7).astype(int)

print(data[['MH_Index','Risk_Level','Sleep_Deficit']].head())
print('\nRisk Level distribution:\n', data['Risk_Level'].value_counts())

   MH_Index Risk_Level  Sleep_Deficit
0      1.33        Low            1.1
1      3.00        Low            0.5
2      2.00        Low            0.9
3      2.67        Low            1.6
4      2.67        Low            3.6

Risk Level distribution:
 Risk_Level
Low       647
Medium    552
High        1
Name: count, dtype: int64


MH_Index represents the overall mental health condition of a teenager, calculated as the average of anxiety, depression, and stress levels. A higher value indicates poorer mental health.

In [50]:
# Interaction features
data['Stress_x_Sleep'] = data['stress_level'] * data['sleep_hours']
data['Social_x_Sleep'] = data['daily_social_media_hours'] * data['sleep_hours']
data['Stress_x_Social'] = data['stress_level'] * data['daily_social_media_hours']

# Lifestyle balance
data['Activity_Ratio'] = data['physical_activity'] / (data['daily_social_media_hours'] + 1)

# Sleep quality
data['Sleep_Quality'] = data['sleep_hours'] / 8.5
print(data[['Stress_x_Sleep','Social_x_Sleep','Stress_x_Social','Activity_Ratio','Sleep_Quality']].head())

   Stress_x_Sleep  Social_x_Sleep  Stress_x_Social  Activity_Ratio  \
0            14.8           58.46             15.8        0.168539   
1            64.0           15.20             15.2        0.275862   
2            15.2            9.88              2.6        0.000000   
3             6.9           51.06              7.4        0.095238   
4            14.7           23.03             14.1        0.245614   

   Sleep_Quality  
0       0.870588  
1       0.941176  
2       0.894118  
3       0.811765  
4       0.576471  


These engineered features capture interactions and imbalances between lifestyle factors. For example, stress combined with low sleep has a stronger impact than either alone. Similarly, ratios like activity to social media usage reflect behavioral patterns. These additions help the model learn more realistic relationships and improve prediction accuracy.

In [51]:
print(data.columns)

Index(['age', 'gender', 'daily_social_media_hours', 'platform_usage',
       'sleep_hours', 'screen_time_before_sleep', 'academic_performance',
       'physical_activity', 'social_interaction_level', 'stress_level',
       'anxiety_level', 'addiction_level', 'depression_label', 'MH_Index',
       'Risk_Level', 'Sleep_Deficit', 'Sleep_Deprived', 'Stress_x_Sleep',
       'Social_x_Sleep', 'Stress_x_Social', 'Activity_Ratio', 'Sleep_Quality'],
      dtype='object')


### 2.3 Selecting Features and Target

In [68]:
data = pd.get_dummies(data, columns=['platform_usage', 'social_interaction_level'], drop_first=True)

In [71]:
target = 'MH_Index'
X = data.drop(['MH_Index', 'Risk_Level'], axis=1)
y = data[target]
print(X.head())

   age  daily_social_media_hours  sleep_hours  screen_time_before_sleep  \
0   14                       7.9          7.4                       2.9   
1   19                       1.9          8.0                       2.9   
2   17                       1.3          7.6                       0.5   
3   15                       7.4          6.9                       1.6   
4   15                       4.7          4.9                       3.0   

   academic_performance  physical_activity  stress_level  anxiety_level  \
0                  3.01                1.5             2              2   
1                  3.22                0.8             8              1   
2                  3.92                0.0             2              4   
3                  3.48                0.8             1              7   
4                  2.37                1.4             3              5   

   addiction_level  depression_label  ...  Stress_x_Sleep  Social_x_Sleep  \
0                1   

In [72]:
print(X.columns)

Index(['age', 'daily_social_media_hours', 'sleep_hours',
       'screen_time_before_sleep', 'academic_performance', 'physical_activity',
       'stress_level', 'anxiety_level', 'addiction_level', 'depression_label',
       'Sleep_Deficit', 'Sleep_Deprived', 'Stress_x_Sleep', 'Social_x_Sleep',
       'Stress_x_Social', 'Activity_Ratio', 'Sleep_Quality', 'gender_male',
       'platform_usage_Instagram', 'platform_usage_TikTok',
       'social_interaction_level_low', 'social_interaction_level_medium'],
      dtype='object')


### 2.4: Feature Scaling

Since input features have different scales, we standardize them 
using StandardScaler. This ensures that features contribute equally 
to the regression model and improves convergence for some algorithms.


In [73]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
print(X_scaled.head())

Y_DF = pd.DataFrame(y, columns=[target])

y_scaler = StandardScaler()
Y_scaled = y_scaler.fit_transform(Y_DF)

Y_scaled = pd.DataFrame(Y_scaled, columns=[target])
print(Y_scaled.head())



        age  daily_social_media_hours  sleep_hours  screen_time_before_sleep  \
0 -0.954099                  1.657833     0.659177                  1.618830   
1  1.519796                 -1.299649     1.075244                  1.618830   
2  0.530238                 -1.595397     0.797866                 -1.731436   
3 -0.459320                  1.411376     0.312455                 -0.195897   
4 -0.459320                  0.080509    -1.074435                  1.758424   

   academic_performance  physical_activity  stress_level  anxiety_level  \
0              0.034026           0.834276     -1.187367      -1.272335   
1              0.398282          -0.368593      0.880116      -1.622199   
2              1.612470          -1.743301     -1.187367      -0.572609   
3              0.849266          -0.368593     -1.531947       0.476980   
4             -1.076088           0.662437     -0.842786      -0.222746   

   addiction_level  depression_label  ...  Stress_x_Sleep  Social_x_